# Phase 6 -- Data Warehousing

**Notebook:** `06_data_warehouse.ipynb`
**Data in:** `../data/ibm_hr_stats_ready.csv`
**Data out:** PostgreSQL database `ibm_hr_db`

---

## What is a Data Warehouse?

A normal database stores raw data for day-to-day operations.
A **data warehouse** stores data that is already organised and ready for reports.

> Normal database = filing cabinet with documents thrown in randomly
> Data warehouse = same documents sorted by topic, date, department — ready to report from

## What is a Star Schema?

The warehouse uses a **Star Schema** — the industry standard for analytical databases.

```
dim_department
      |
dim_job -- fact_attrition -- dim_employee
```

| Table            | Type          | Contains                             | Rows   |
| ---------------- | ------------- | ------------------------------------ | ------ |
| `fact_attrition` | Fact (centre) | Numbers: income, overtime, attrition | 1470   |
| `dim_department` | Dimension     | Department names                     | 3      |
| `dim_job`        | Dimension     | Job role + level                     | varies |
| `dim_employee`   | Dimension     | Age, gender, education               | 1470   |

**Why 4 tables instead of 1?**
Instead of repeating "Research & Development" 961 times, we store it once and use a small ID number (`dept_id=1`). Faster queries, no typos, smaller storage.

## PostgreSQL Setup (run once)

```sql
-- In pgAdmin or psql:
CREATE DATABASE ibm_hr_db;
```

```bash
-- In terminal:
pip install psycopg2-binary sqlalchemy pandas
```


## Step 1 -- Connect to PostgreSQL

**What this does:** Creates the connection to your PostgreSQL database.
Change `DB_PASSWORD` to match your installation.

All cells below use this `engine` variable to talk to the database.


In [1]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text
import warnings
warnings.filterwarnings('ignore')

# Change DB_PASSWORD to match your PostgreSQL installation
DB_USER     = 'postgres'
DB_PASSWORD = 'admin'
DB_HOST     = 'localhost'
DB_PORT     = '5432'
DB_NAME     = 'ibm_hr_db'

engine = create_engine(
    f'postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}'
)

try:
    with engine.connect() as conn:
        conn.execute(text('SELECT 1'))
    print("Connected to PostgreSQL")
    print(f"Database: {DB_NAME}")
except Exception as e:
    print(f"Connection failed: {e}")
    print("Please update DB_PASSWORD with your correct PostgreSQL password.")
print("Connected to PostgreSQL")
print(f"Database: {DB_NAME}")

df = pd.read_csv('../data/ibm_hr_stats_ready.csv')
print(f"Loaded  : {df.shape[0]} rows x {df.shape[1]} columns")
print(f"Attrition rate: {df['Attrition'].mean()*100:.2f}%")

Connected to PostgreSQL
Database: ibm_hr_db
Connected to PostgreSQL
Database: ibm_hr_db
Loaded  : 1470 rows x 39 columns
Attrition rate: 16.12%


**Expected output:**

```
Connected to PostgreSQL
Database: ibm_hr_db
Loaded  : 1470 rows x 39 columns
Attrition rate: 16.12%
```

If connection fails: check PostgreSQL is running, password is correct, and `ibm_hr_db` database was created in pgAdmin.


## Step 2 -- Create the 4 Tables (DDL)

**What this does:** Creates the 4 empty tables in PostgreSQL.
`IF NOT EXISTS` = safe to re-run.

**DDL (Data Definition Language)** = SQL that creates structure, not data.
**DML (Data Manipulation Language)** = SQL that inserts/updates data (next step).


In [4]:
# DDL = Data Definition Language = SQL that creates table structure
# IF NOT EXISTS = safe to re-run, will not crash if table already exists

ddl = [

    # Dimension 1: only 3 rows (Sales, HR, R&D)
    'CREATE TABLE IF NOT EXISTS dim_department (dept_id SERIAL PRIMARY KEY, dept_name VARCHAR(100) NOT NULL UNIQUE)',

    # Dimension 2: job role + level combinations
    'CREATE TABLE IF NOT EXISTS dim_job (job_id SERIAL PRIMARY KEY, job_role VARCHAR(100), job_level INTEGER)',

    # Dimension 3: who the employee is (personal details)
    'CREATE TABLE IF NOT EXISTS dim_employee (employee_id INTEGER PRIMARY KEY, age INTEGER, gender VARCHAR(10), marital_status VARCHAR(20), business_travel VARCHAR(20), education INTEGER, education_field VARCHAR(50))',

    # Fact table: all the measurable numbers (centre of the star)
    '''CREATE TABLE IF NOT EXISTS fact_attrition (
        employee_id              INTEGER REFERENCES dim_employee(employee_id),
        dept_id                  INTEGER REFERENCES dim_department(dept_id),
        job_id                   INTEGER REFERENCES dim_job(job_id),
        monthly_income           INTEGER,
        annual_income            INTEGER,
        years_at_company         INTEGER,
        total_working_years      INTEGER,
        overtime                 INTEGER,
        job_satisfaction         INTEGER,
        environment_satisfaction INTEGER,
        work_life_balance        INTEGER,
        stock_option_level       INTEGER,
        distance_from_home       INTEGER,
        years_since_last_promo   INTEGER,
        satisfaction_score       FLOAT,
        is_high_risk             INTEGER,
        below_dept_avg           INTEGER,
        income_zscore            FLOAT,
        attrition                INTEGER
    )'''
]

with engine.begin() as conn:
    for stmt in ddl:
        conn.execute(text(stmt))

print("4 tables created:")
print("  dim_department  (3 unique departments)")
print("  dim_job         (role + level combinations)")
print("  dim_employee    (1470 employee personal records)")
print("  fact_attrition  (1470 rows of numbers - the centre of the star)")

4 tables created:
  dim_department  (3 unique departments)
  dim_job         (role + level combinations)
  dim_employee    (1470 employee personal records)
  fact_attrition  (1470 rows of numbers - the centre of the star)


**The `REFERENCES` keyword creates foreign keys.**
`dept_id INTEGER REFERENCES dim_department(dept_id)` means every dept_id in the fact table must exist in dim_department. PostgreSQL enforces this automatically — it prevents orphaned records.

Dimension tables have no REFERENCES — they are independent. Fact table points to all three dimensions.


## Step 3 -- Load the 3 Dimension Tables

**What this does:** Fills dim_department, dim_job, dim_employee with their data.

We load dimension tables FIRST because the fact table needs their IDs.
`drop_duplicates()` ensures each table has only unique values — not 1470 copies.


In [6]:
# ── dim_department ────────────────────────────────────────────────────
dept_df = df[['Department']].drop_duplicates().reset_index(drop=True)
dept_df['dept_id'] = dept_df.index + 1
dept_df = dept_df.rename(columns={'Department':'dept_name'})[['dept_id','dept_name']]
with engine.begin() as conn:
    conn.execute(text('DROP VIEW IF EXISTS v_dept_summary, v_role_summary, v_high_risk_list'))
dept_df.to_sql('dim_department', engine, if_exists='replace', index=False)
print("dim_department loaded:")
print(dept_df.to_string(index=False))

# ── dim_job ───────────────────────────────────────────────────────────
job_df = df[['JobRole','JobLevel']].drop_duplicates().reset_index(drop=True)
job_df['job_id'] = job_df.index + 1
job_df = job_df.rename(columns={'JobRole':'job_role','JobLevel':'job_level'})[['job_id','job_role','job_level']]
job_df.to_sql('dim_job', engine, if_exists='replace', index=False)
print(f"\ndim_job loaded: {len(job_df)} rows")
print(job_df.to_string(index=False))

# ── dim_employee ───────────────────────────────────────────────────────
emp_df = df[['EmployeeNumber','Age','Gender','MaritalStatus','BusinessTravel','Education','EducationField']].copy()
emp_df.columns = ['employee_id','age','gender','marital_status','business_travel','education','education_field']
emp_df.to_sql('dim_employee', engine, if_exists='replace', index=False)
print(f"\ndim_employee loaded: {len(emp_df)} rows")
print(emp_df.head(3).to_string(index=False))

dim_department loaded:
 dept_id              dept_name
       1                  Sales
       2 Research & Development
       3        Human Resources

dim_job loaded: 26 rows
 job_id                  job_role  job_level
      1           Sales Executive          2
      2        Research Scientist          2
      3     Laboratory Technician          1
      4        Research Scientist          1
      5    Manufacturing Director          3
      6 Healthcare Representative          2
      7     Laboratory Technician          2
      8                   Manager          4
      9    Manufacturing Director          2
     10      Sales Representative          1
     11         Research Director          3
     12                   Manager          5
     13 Healthcare Representative          3
     14      Sales Representative          2
     15           Sales Executive          3
     16         Research Director          5
     17     Laboratory Technician          3
     18       

**Why `drop_duplicates()`?**
There are 3 unique departments across 1470 employees. `drop_duplicates()` collapses all 1470 rows to just 3 unique department names.
Same logic for job roles — there are only 9 unique roles across 1470 employees.

This is the core of the star schema concept: descriptive data stored once, referenced many times.


## Step 4 -- Load the Fact Table

**What this does:** Fills fact_attrition with all the measurable columns.
We merge with dimension tables first to get the foreign key IDs (`dept_id`, `job_id`).

This is the most important step — the fact table is the centre of the star.


In [7]:
# Step 1: merge to get dept_id and job_id for each employee
df_merged = df.merge(
    dept_df.rename(columns={'dept_name':'Department'}), on='Department', how='left'
).merge(
    job_df.rename(columns={'job_role':'JobRole','job_level':'JobLevel'}), on=['JobRole','JobLevel'], how='left'
)

# Step 2: select and rename only the fact columns
fact_df = df_merged[[
    'EmployeeNumber', 'dept_id', 'job_id',
    'MonthlyIncome', 'AnnualIncome',
    'YearsAtCompany', 'TotalWorkingYears',
    'OverTime', 'JobSatisfaction', 'EnvironmentSatisfaction',
    'WorkLifeBalance', 'StockOptionLevel', 'DistanceFromHome',
    'YearsSinceLastPromotion', 'SatisfactionScore',
    'IsHighRisk', 'Below_Dept_Avg', 'Income_Zscore', 'Attrition'
]].copy()

fact_df.columns = [
    'employee_id', 'dept_id', 'job_id',
    'monthly_income', 'annual_income',
    'years_at_company', 'total_working_years',
    'overtime', 'job_satisfaction', 'environment_satisfaction',
    'work_life_balance', 'stock_option_level', 'distance_from_home',
    'years_since_last_promo', 'satisfaction_score',
    'is_high_risk', 'below_dept_avg', 'income_zscore', 'attrition'
]

fact_df.to_sql('fact_attrition', engine, if_exists='replace', index=False)
print(f"fact_attrition loaded: {fact_df.shape[0]} rows x {fact_df.shape[1]} columns")
print()
print("All 4 tables loaded. Warehouse is ready.")

fact_attrition loaded: 1470 rows x 19 columns

All 4 tables loaded. Warehouse is ready.


**Why merge first?**
The fact table stores `dept_id=1` (not "Sales"). To assign the correct ID, we merge with `dim_department` which tells us: Sales = dept_id 1, HR = dept_id 2, R&D = dept_id 3.
Then the fact table stores that small integer instead of the full text string.

`if_exists='replace'` drops and recreates the table — makes the notebook safe to re-run from the beginning.


## Step 5 -- Validate the Warehouse

**What this does:** Runs JOIN queries across all 4 tables.
If the numbers match Phase 1 exactly, the star schema was built correctly.

**This is critical.** A wrong JOIN silently returns wrong numbers — always validate.


In [8]:
def wh(query, label=''):
    result = pd.read_sql(query, engine)
    if label:
        print(f'\n=== {label} ===')
    print(result.to_string(index=False))
    return result
# Must return 16.12% to prove the load worked
wh('''
    SELECT COUNT(*) AS total,
           SUM(attrition) AS left_company,
           ROUND(AVG(attrition::NUMERIC) * 100.0, 2) AS attrition_pct
    FROM fact_attrition
''', 'Validation 1: Overall rate -- must equal 16.12%')

# Must show Sales 20.6%, HR 19.0%, R&D 13.8% (same as Phase 5 Q5)
wh('''
    SELECT d.dept_name,
           COUNT(*) AS employees,
           ROUND(AVG(f.attrition::NUMERIC) * 100.0, 1) AS attrition_pct,
           ROUND(AVG(f.monthly_income), 0) AS avg_income
    FROM fact_attrition f
    JOIN dim_department d ON f.dept_id = d.dept_id
    GROUP BY d.dept_name
    ORDER BY attrition_pct DESC
''', 'Validation 2: By Dept via JOIN -- must match Phase 5 Q5')

# Must show Sales Rep 39.8%, Lab Tech 23.9% (same as Phase 5 Q6)
wh('''
    SELECT j.job_role,
           COUNT(*) AS employees,
           ROUND(AVG(f.attrition::NUMERIC) * 100.0, 1) AS attrition_pct,
           ROUND(AVG(f.monthly_income), 0) AS avg_income
    FROM fact_attrition f
    JOIN dim_job j ON f.job_id = j.job_id
    GROUP BY j.job_role
    ORDER BY attrition_pct DESC
    LIMIT 5
''', 'Validation 3: By Role via JOIN -- must match Phase 5 Q6')


=== Validation 1: Overall rate -- must equal 16.12% ===
 total  left_company  attrition_pct
  1470         237.0          16.12

=== Validation 2: By Dept via JOIN -- must match Phase 5 Q5 ===
             dept_name  employees  attrition_pct  avg_income
                 Sales        446           20.6      6959.0
       Human Resources         63           19.0      6655.0
Research & Development        961           13.8      6281.0

=== Validation 3: By Role via JOIN -- must match Phase 5 Q6 ===
             job_role  employees  attrition_pct  avg_income
 Sales Representative         83           39.8      2626.0
Laboratory Technician        259           23.9      3237.0
      Human Resources         52           23.1      4236.0
      Sales Executive        326           17.5      6924.0
   Research Scientist        292           16.1      3240.0


,job_role,employees,attrition_pct,avg_income
0,Sales Representative,83,39.8,2626.0
1,Laboratory Technician,259,23.9,3237.0
2,Human Resources,52,23.1,4236.0
3,Sales Executive,326,17.5,6924.0
4,Research Scientist,292,16.1,3240.0


**All 3 validations must match previous phases exactly:**

- Validation 1: 16.12% — matches Phase 1 EDA and Phase 5 Q2
- Validation 2: Sales 20.6%, HR 19.0%, R&D 13.8% — matches Phase 5 Q5
- Validation 3: Sales Rep 39.8%, Lab Tech 23.9% — matches Phase 5 Q6

**PostgreSQL note:** `attrition::FLOAT` is PostgreSQL casting syntax. It is equivalent to `CAST(attrition AS FLOAT)` in SQLite. This is why Phase 6 requires PostgreSQL — it uses native PG syntax throughout.


## Step 6 -- Create Views in PostgreSQL

**What this does:** Creates 3 saved SQL queries (Views).
`CREATE OR REPLACE VIEW` works correctly in PostgreSQL — this was the error in Phase 5 which used SQLite.

**Why views?** Power BI connects to these views. Non-technical users can query them. If the data changes, views update automatically.


In [9]:

# PostgreSQL supports CREATE OR REPLACE VIEW (unlike SQLite which gave an error in Phase 5)
# These 3 views map directly to 3 pages in the Phase 7 Power BI dashboard

view_sql = [

    # View 1: Department summary -- Power BI Overview page
    '''CREATE OR REPLACE VIEW v_dept_summary AS
    SELECT d.dept_name,
           COUNT(*) AS total_employees,
           SUM(f.attrition) AS employees_left,
           ROUND(AVG(f.attrition::NUMERIC) * 100.0, 1) AS attrition_pct,
           ROUND(AVG(f.monthly_income), 0) AS avg_income,
           SUM(f.is_high_risk) AS high_risk_count,
           SUM(f.overtime) AS overtime_count
    FROM fact_attrition f
    JOIN dim_department d ON f.dept_id = d.dept_id
    GROUP BY d.dept_name
    ORDER BY attrition_pct DESC''',

    # View 2: Role breakdown -- Power BI Role Analysis page
    '''CREATE OR REPLACE VIEW v_role_summary AS
    SELECT j.job_role, j.job_level, d.dept_name,
           COUNT(*) AS total_employees,
           ROUND(AVG(f.attrition::NUMERIC) * 100.0, 1) AS attrition_pct,
           ROUND(AVG(f.monthly_income), 0) AS avg_income,
           ROUND(AVG(f.job_satisfaction::NUMERIC), 2) AS avg_satisfaction,
           SUM(f.overtime) AS overtime_count
    FROM fact_attrition f
    JOIN dim_job j ON f.job_id = j.job_id
    JOIN dim_department d ON f.dept_id = d.dept_id
    GROUP BY j.job_role, j.job_level, d.dept_name
    ORDER BY attrition_pct DESC''',

    # View 3: HR intervention list -- Power BI Action page
    '''CREATE OR REPLACE VIEW v_high_risk_list AS
    SELECT f.employee_id, e.age, e.marital_status,
           d.dept_name, j.job_role,
           f.monthly_income, f.years_at_company,
           f.overtime, f.job_satisfaction,
           f.stock_option_level, f.is_high_risk, f.attrition
    FROM fact_attrition f
    JOIN dim_employee e ON f.employee_id = e.employee_id
    JOIN dim_department d ON f.dept_id = d.dept_id
    JOIN dim_job j ON f.job_id = j.job_id
    WHERE f.is_high_risk = 1
    ORDER BY f.monthly_income ASC'''
]

with engine.connect() as conn:
    for sql in view_sql:
        conn.execute(text(sql))
    conn.commit()

print("3 views created:")
print("  v_dept_summary   -- for Power BI overview page")
print("  v_role_summary   -- for Power BI role analysis page")
print("  v_high_risk_list -- for Power BI HR intervention page")
print()

# Preview each view
for name in ['v_dept_summary','v_role_summary','v_high_risk_list']:
    result = pd.read_sql(f'SELECT * FROM {name} LIMIT 3', engine)
    print(f"=== {name} ===")
    print(result.to_string(index=False))
    print()

3 views created:
  v_dept_summary   -- for Power BI overview page
  v_role_summary   -- for Power BI role analysis page
  v_high_risk_list -- for Power BI HR intervention page

=== v_dept_summary ===
             dept_name  total_employees  employees_left  attrition_pct  avg_income  high_risk_count  overtime_count
                 Sales              446            92.0           20.6      6959.0             40.0           128.0
       Human Resources               63            12.0           19.0      6655.0              9.0            17.0
Research & Development              961           133.0           13.8      6281.0            104.0           271.0

=== v_role_summary ===
             job_role  job_level              dept_name  total_employees  attrition_pct  avg_income  avg_satisfaction  overtime_count
 Sales Representative          1                  Sales               76           42.1      2507.0              2.67            22.0
Laboratory Technician          3 Research & 

**3 views = 3 Power BI dashboard pages:**

| View               | Dashboard page     | What it shows                         |
| ------------------ | ------------------ | ------------------------------------- |
| `v_dept_summary`   | Overview page      | KPI cards by department               |
| `v_role_summary`   | Role Analysis page | Bar charts by job role                |
| `v_high_risk_list` | HR Action page     | Individual employee intervention list |

In Power BI: `Home -> Get Data -> PostgreSQL -> ibm_hr_db -> select views`


## Step 7 -- Export CSVs for Power BI

**What this does:** Saves all tables and views as CSV files.
Use these if you cannot connect Power BI directly to PostgreSQL.

`Home -> Get Data -> Text/CSV` in Power BI Desktop.


In [10]:
import os
os.makedirs('../data/warehouse_exports', exist_ok=True)

# Export all tables and views as CSVs (fallback if Power BI cannot connect to PostgreSQL)
exports = {
    'fact_attrition'  : 'SELECT * FROM fact_attrition',
    'dim_department'  : 'SELECT * FROM dim_department',
    'dim_job'         : 'SELECT * FROM dim_job',
    'dim_employee'    : 'SELECT * FROM dim_employee',
    'v_dept_summary'  : 'SELECT * FROM v_dept_summary',
    'v_role_summary'  : 'SELECT * FROM v_role_summary',
    'v_high_risk_list': 'SELECT * FROM v_high_risk_list',
}

print("Exporting to ../data/warehouse_exports/")
for name, query in exports.items():
    df_exp = pd.read_sql(query, engine)
    df_exp.to_csv(f'../data/warehouse_exports/{name}.csv', index=False)
    print(f"  {name}.csv  ({df_exp.shape[0]} rows x {df_exp.shape[1]} cols)")

print()
print("Power BI connection options:")
print("  Option A (best): Home -> Get Data -> PostgreSQL -> ibm_hr_db")
print("  Option B: Home -> Get Data -> each CSV -> then Model view for relationships")

Exporting to ../data/warehouse_exports/
  fact_attrition.csv  (1470 rows x 19 cols)
  dim_department.csv  (3 rows x 2 cols)
  dim_job.csv  (26 rows x 3 cols)
  dim_employee.csv  (1470 rows x 7 cols)
  v_dept_summary.csv  (3 rows x 7 cols)
  v_role_summary.csv  (31 rows x 8 cols)
  v_high_risk_list.csv  (153 rows x 12 cols)

Power BI connection options:
  Option A (best): Home -> Get Data -> PostgreSQL -> ibm_hr_db
  Option B: Home -> Get Data -> each CSV -> then Model view for relationships


## Phase 6 -- Summary

**What was built:**

| Item                      | Description                         |
| ------------------------- | ----------------------------------- |
| `dim_department`          | 3 rows — department lookup          |
| `dim_job`                 | Job role + level combinations       |
| `dim_employee`            | 1470 rows — personal details        |
| `fact_attrition`          | 1470 rows, 19 columns — all numbers |
| `v_dept_summary`          | View for Power BI overview          |
| `v_role_summary`          | View for Power BI role page         |
| `v_high_risk_list`        | View for Power BI action page       |
| `warehouse_exports/*.csv` | Flat file fallback                  |

**Project data lineage:**

```
ibm_hr_raw.csv (35 cols)
    -> Phase 4 pipeline.py
ibm_hr_stats_ready.csv (39 cols)
    -> Phase 6 this notebook
PostgreSQL ibm_hr_db
    -> Phase 7 Power BI
Dashboard (4 pages)
```

**Next: Phase 7 — Power BI Dashboard**
